<a href="https://colab.research.google.com/github/Itzcskh/genai-bootcamp-lab1/blob/main/%D9%85%D9%8F%D8%B1%D9%8E%D9%86%D9%90%D9%91%D9%82_%7C_Muranniq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q --no-warn-conflicts transformers>=4.56 accelerate sentence-transformers gradio soundfile gTTS huggingface_hub
print("Step 1 Complete: All libraries installed successfully.")

In [ ]:
from google.colab import files
import os

print("Please select and upload your presentation audio file (MP3 or WAV):")
uploaded = files.upload()

if uploaded:
    audio_filename = list(uploaded.keys())[0]
    print(f"\nStep 2 Complete: Audio file '{audio_filename}' successfully uploaded and ready.")
else:
    print("Error: No file uploaded. Please run the cell again and choose a file.")

In [ ]:
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using Device: {device}")

print("Loading OpenAI Whisper Large V3 Turbo (Audio Transcriber)...")
asr_pipeline = pipeline(
    task="automatic-speech-recognition",
    model="openai/whisper-large-v3-turbo",
    chunk_length_s=30,
    device=device
)

print("Loading Qwen 2.5 1.5B Instruct (Behavioral Analyst)...")
llm_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(llm_id)
llm = AutoModelForCausalLM.from_pretrained(
    llm_id,
    dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto"
)
print("Step 3 Complete: Both models are loaded and ready.")

In [ ]:
import numpy as np
import re

if 'audio_filename' in locals() and os.path.exists(audio_filename):
    print(f"Processing audio: {audio_filename}...")

    # Transcribe audio
    asr_result = asr_pipeline(audio_filename, return_timestamps=True, generate_kwargs={"language": "ar"})
    raw_text = asr_result["text"]
    chunks = asr_result["chunks"]

    # Calculate metrics
    pauses = 0
    durations = []
    times_axis = []

    for i, chunk in enumerate(chunks):
        if chunk["timestamp"][0] is not None and chunk["timestamp"][1] is not None:
            times_axis.append(chunk["timestamp"][0])
            segment_len = max(0.1, chunk["timestamp"][1] - chunk["timestamp"][0])
            durations.append(len(chunk["text"]) / segment_len)

            if i > 0 and chunks[i-1]["timestamp"][1] is not None:
                silence_gap = chunk["timestamp"][0] - chunks[i-1]["timestamp"][1]
                if silence_gap > 1.2:
                    pauses += 1

    total_duration = chunks[-1]["timestamp"][1] if chunks and chunks[-1]["timestamp"][1] else 0

    # Preprocess and clean text to prevent repetition issues
    cleaned_text = re.sub(r'\b(و|وا|في)\b(?:\s+\b\1\b)+', r'\1', raw_text)
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()

    words = cleaned_text.split()
    if len(words) > 120:
        cleaned_text = " ".join(words[:120]) + "..."

    print("\nStep 4 Complete: Speech metrics extracted successfully.")
    print(f"Total Duration: {total_duration:.1f}s")
    print(f"Detected Pauses: {pauses}")
    print(f"Cleaned Text: {cleaned_text}")
else:
    print("Error: Please upload your audio file in Step 2 first.")

In [ ]:
import gradio as gr
import numpy as np
import re
import plotly.graph_objects as go

def analyze_presentation_local(audio_path, consent_given):
    if not consent_given:
        # Return an empty figure and the error message
        empty_fig = go.Figure()
        empty_fig.update_layout(title="No data available")
        return "Error: You must accept the data privacy agreement in the sidebar to proceed with the analysis.", empty_fig

    if audio_path is None:
        empty_fig = go.Figure()
        empty_fig.update_layout(title="No data available")
        return "Error: Please upload an audio file or record your voice to start.", empty_fig

    try:
        # 1. Audio Transcription
        asr_result = asr_pipeline(audio_path, return_timestamps=True, generate_kwargs={"language": "ar"})
        raw_text = asr_result["text"]
        chunks = asr_result["chunks"]

        # 2. Metric & Timeline Calculation
        pauses = 0
        durations = []
        times_axis = []

        for i, chunk in enumerate(chunks):
            if chunk["timestamp"][0] is not None and chunk["timestamp"][1] is not None:
                start_time = chunk["timestamp"][0]
                end_time = chunk["timestamp"][1]
                times_axis.append(start_time)

                segment_len = max(0.1, end_time - start_time)
                speech_density = len(chunk["text"]) / segment_len
                durations.append(speech_density)

                if i > 0 and chunks[i-1]["timestamp"][1] is not None:
                    silence_gap = start_time - chunks[i-1]["timestamp"][1]
                    if silence_gap > 1.2:
                        pauses += 1

        total_duration = chunks[-1]["timestamp"][1] if chunks and chunks[-1]["timestamp"][1] else 0
        avg_speech_rate = np.mean(durations) if durations else 0

        # 3. Create Interactive Plotly Timeline
        fig = go.Figure()
        if times_axis:
            fig.add_trace(go.Scatter(
                x=times_axis,
                y=durations,
                mode='lines+markers',
                name='Speech Density',
                line=dict(color='#008080', width=3),
                marker=dict(size=8, color='#004d40')
            ))
            fig.update_layout(
                title="Speech Flow Timeline (Vocal Pacing Tracker)",
                xaxis_title="Time (seconds)",
                yaxis_title="Speaking Rate (characters per second)",
                template="plotly_white",
                height=350,
                margin=dict(l=40, r=40, t=40, b=40)
            )
        else:
            fig.update_layout(title="No timeline data extracted")

        # 4. Clean Text
        cleaned_text = re.sub(r'\b(و|وا|في)\b(?:\s+\b\1\b)+', r'\1', raw_text)
        cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()

        words = cleaned_text.split()
        if len(words) > 120:
            cleaned_text = " ".join(words[:120]) + "..."

        # 5. Prompt & Evaluation Generation
        system_prompt = """
        You are an expert academic advisor and a Human-Computer Interaction (HCI) speech specialist at Najran University.
        Your task is to analyze the transcribed Arabic presentation script of a university student and provide a highly professional, constructive, and encouraging evaluation report in simple, clear English.

        Format your evaluation report strictly under the following numbered headers:

        1. Speech Flow and Vocal Filler Analysis
        - Identify local vocal fillers (such as 'yaani', 'ummm', 'tayyeb') and describe their impact on audience engagement.
        - Evaluate the naturalness and placement of the pauses detected in the speech.

        2. Structural Organization and Idea Coherence
        - Analyze the logical transition from the introduction to the main technical thesis.
        - Evaluate the general semantic cohesion of the presentation.

        3. Strategic Action Plan (HCI Recommendations)
        - Provide three concrete, actionable, and leadership-driven tips for the student to improve their vocal range, pitch variance, and pacing in future presentations.

        Important: Be concise, direct, and academic. Do not repeat sentences or use emojis.
        """

        user_content = f"Analyze the following transcribed text:\n\"{cleaned_text}\""
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content}
        ]

        model_inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(llm.device)
        outputs = llm.generate(**model_inputs, max_new_tokens=450, pad_token_id=tokenizer.eos_token_id, do_sample=False, repetition_penalty=1.35)
        input_len = model_inputs[list(model_inputs.keys())[0]].shape[1]
        evaluation_report = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()

        final_markdown_output = f"""
# Emotion Mirror Presentation Analytics

## 1. Quantitative Performance Metrics
- Total Speech Duration: {total_duration:.1f} seconds
- Identified Hesitation Pauses: {pauses} pauses
- Average Verbal Density: {avg_speech_rate:.1f} characters/second

## 2. Transcribed Speech (Whisper Model)
"{cleaned_text}"

## 3. Professional HCI Evaluation Report
{evaluation_report}
"""
        return final_markdown_output, fig

    except Exception as e:
        empty_fig = go.Figure()
        empty_fig.update_layout(title="Analysis Error")
        return f"System Error: {str(e)}", empty_fig

# Gradio Dashboard Design
with gr.Blocks(title="مُرَنِّق | Muranniq - Speech Multimodal Analysis") as demo:

    # 1. Branding Header Area
    gr.Markdown("""
    <div style="text-align: center; padding: 15px; border-bottom: 2px solid #008080;">
        <h1 style="color: #008080; margin-bottom: 5px; font-family: 'Times New Roman', serif;">مُرَنِّق | Muranniq</h1>
        <h3 style="color: #555; margin-top: 0; font-weight: normal; font-style: italic;">
            &bull; Refining Speech, Elevating Performance
        </h3>
        <p style="color: #777; font-size: 14px;">
            An Academic Evaluation & Vocal Calibration System | Najran University Multimodal Capstone Project
        </p>
    </div>
    """)

    # 2. Interactive Columns
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Configuration and Inputs")
            consent_checkbox = gr.Checkbox(
                label="I agree to let the system process this audio locally for academic evaluation.",
                value=False
            )
            audio_uploader = gr.Audio(
                label="Upload Presentation Audio (WAV or MP3)",
                type="filepath"
            )
            submit_button = gr.Button("Analyze Presentation", variant="primary")

        with gr.Column(scale=2):
            gr.Markdown("### System Analytics Output")
            timeline_plot = gr.Plot(label="Speech Pace Over Time")
            output_report = gr.Markdown("### Waiting for inputs...")

    # 3. Action Click Binder
    submit_button.click(
        fn=analyze_presentation_local,
        inputs=[audio_uploader, consent_checkbox],
        outputs=[output_report, timeline_plot]
    )

# Launch local interface
demo.launch(inline=True, share=True)

In [ ]:
import os

# 1. Write requirements.txt safely to disk
requirements_text = """transformers>=4.56
accelerate
sentence-transformers
gradio
soundfile
gTTS
torch
plotly
"""

with open("requirements.txt", "w") as req_file:
    req_file.write(requirements_text)

print("Successfully generated: requirements.txt")

# 2. Write app.py safely to disk (incorporating your exact Step 5.5 code)
app_lines = [
    "import os\n",
    "import re\n",
    "import torch\n",
    "import gradio as gr\n",
    "import numpy as np\n",
    "import plotly.graph_objects as go\n",
    "from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer\n\n",
    'device = "cuda" if torch.cuda.is_available() else "cpu"\n\n',
    "asr_pipeline = pipeline(\n",
    '    task="automatic-speech-recognition",\n',
    '    model="openai/whisper-large-v3-turbo",\n',
    "    chunk_length_s=30,\n",
    "    device=device\n",
    ")\n\n",
    'llm_id = "Qwen/Qwen2.5-1.5B-Instruct"\n',
    "tokenizer = AutoTokenizer.from_pretrained(llm_id)\n",
    "llm = AutoModelForCausalLM.from_pretrained(\n",
    "    llm_id,\n",
    "    dtype=torch.float16 if device == 'cuda' else torch.float32,\n",
    '    device_map="auto"\n',
    ")\n\n",
    "def analyze_presentation(audio_path, consent_given):\n",
    "    if not consent_given:\n",
    "        empty_fig = go.Figure()\n",
    '        empty_fig.update_layout(title="No data available")\n',
    '        return "Error: You must accept the data privacy agreement in the sidebar to proceed with the analysis.", empty_fig\n\n',
    "    if audio_path is None:\n",
    "        empty_fig = go.Figure()\n",
    '        empty_fig.update_layout(title="No data available")\n',
    '        return "Error: Please upload an audio file or record your voice to start.", empty_fig\n\n',
    "    try:\n",
    '        asr_result = asr_pipeline(audio_path, return_timestamps=True, generate_kwargs={"language": "ar"})\n',
    '        raw_text = asr_result["text"]\n',
    '        chunks = asr_result["chunks"]\n\n',
    "        pauses = 0\n",
    "        durations = []\n",
    "        times_axis = []\n\n",
    "        for i, chunk in enumerate(chunks):\n",
    '            if chunk["timestamp"][0] is not None and chunk["timestamp"][1] is not None:\n',
    '                start_time = chunk["timestamp"][0]\n',
    '                end_time = chunk["timestamp"][1]\n',
    "                times_axis.append(start_time)\n",
    "                segment_len = max(0.1, end_time - start_time)\n",
    '                speech_density = len(chunk["text"]) / segment_len\n',
    "                durations.append(speech_density)\n\n",
    '                if i > 0 and chunks[i-1]["timestamp"][1] is not None:\n',
    '                    silence_gap = start_time - chunks[i-1]["timestamp"][1]\n',
    "                    if silence_gap > 1.2:\n",
    "                        pauses += 1\n\n",
    '        total_duration = chunks[-1]["timestamp"][1] if chunks and chunks[-1]["timestamp"][1] else 0\n',
    "        avg_speech_rate = np.mean(durations) if durations else 0\n\n",
    "        fig = go.Figure()\n",
    "        if times_axis:\n",
    "            fig.add_trace(go.Scatter(\n",
    "                x=times_axis,\n",
    "                y=durations,\n",
    "                mode='lines+markers',\n",
    "                name='Speech Density',\n",
    "                line=dict(color='#008080', width=3),\n",
    "                marker=dict(size=8, color='#004d40')\n",
    "            ))\n",
    "            fig.update_layout(\n",
    "                title='Speech Flow Timeline (Vocal Pacing Tracker)',\n",
    "                xaxis_title='Time (seconds)',\n",
    "                yaxis_title='Speaking Rate (characters per second)',\n",
    "                template='plotly_white',\n",
    "                height=350,\n",
    "                margin=dict(l=40, r=40, t=40, b=40)\n",
    "            )\n",
    "        else:\n",
    "            fig.update_layout(title='No timeline data extracted')\n\n",
    "        cleaned_text = re.sub(r'\\\\b(و|وا|في)\\\\b(?:\\\\s+\\\\b\\\\1\\\\b)+', r'\\\\1', raw_text)\n",
    "        cleaned_text = re.sub(r'\\\\s+', ' ', cleaned_text).strip()\n\n",
    "        words = cleaned_text.split()\n",
    "        if len(words) > 120:\n",
    "            cleaned_text = ' '.join(words[:120]) + '...'\n\n",
    '        system_prompt = """You are an expert academic advisor and a Human-Computer Interaction (HCI) speech specialist at Najran University.\\nYour task is to analyze the transcribed Arabic presentation script of a university student and provide a highly professional, constructive, and encouraging evaluation report in simple, clear English.\\n\\nFormat your evaluation report strictly under the following numbered headers:\\n\\n1. Speech Flow and Vocal Filler Analysis\\n- Identify local vocal fillers (such as \'yaani\', \'ummm\', \'tayyeb\') and describe their impact on audience engagement.\\n- Evaluate the naturalness and placement of the pauses detected in the speech.\\n\\n2. Structural Organization and Idea Coherence\\n- Analyze the logical transition from the introduction to the main technical thesis.\\n- Evaluate the general semantic cohesion of the presentation.\\n\\n3. Strategic Action Plan (HCI Recommendations)\\n- Provide three concrete, actionable, and leadership-driven tips for the student to improve their vocal range, pitch variance, and pacing in future presentations.\\n\\nImportant: Be concise, direct, and academic. Do not repeat sentences or use emojis."""\n\n',
    '        user_content = f"Analyze the following transcribed text:\\\\n\\\\\\"{cleaned_text}\\\\\\""\n',
    '        messages = [\n',
    '            {"role": "system", "content": system_prompt},\n',
    '            {"role": "user", "content": user_content}\n',
    '        ]\n',
    '        model_inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(llm.device)\n',
    "        outputs = llm.generate(**model_inputs, max_new_tokens=450, pad_token_id=tokenizer.eos_token_id, do_sample=False, repetition_penalty=1.35)\n",
    "        input_len = model_inputs[list(model_inputs.keys())[0]].shape[1]\n",
    "        evaluation_report = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()\n\n",
    '        final_markdown_output = f"""# Emotion Mirror Presentation Analytics\\n\\n## 1. Quantitative Performance Metrics\\n- Total Speech Duration: {total_duration:.1f} seconds\\n- Identified Hesitation Pauses: {pauses} pauses\\n- Average Verbal Density: {avg_speech_rate:.1f} characters/second\\n\\n## 2. Transcribed Speech (Whisper Model)\\n\\\\\\"{cleaned_text}\\\\\\"\\n\\n## 3. Professional HCI Evaluation Report\\n{evaluation_report}"""\n',
    "        return final_markdown_output, fig\n",
    "    except Exception as e:\n",
    "        empty_fig = go.Figure()\n",
    '        empty_fig.update_layout(title="Analysis Error")\n',
    '        return f"System Error: {str(e)}", empty_fig\n\n',
    'with gr.Blocks(title="مُرَنِّق | Muranniq - Speech Multimodal Analysis") as demo:\n',
    '    gr.Markdown("""\n',
    '    <div style="text-align: center; padding: 15px; border-bottom: 2px solid #008080;">\n',
    '        <h1 style="color: #008080; margin-bottom: 5px; font-family: \'Times New Roman\', serif;">مُرَنِّق | Muranniq</h1>\n',
    '        <h3 style="color: #555; margin-top: 0; font-weight: normal; font-style: italic;">\n',
    "            &bull; Refining Speech, Elevating Performance\n",
    "        </h3>\n",
    '        <p style="color: #777; font-size: 14px;">\n',
    "            An Academic Evaluation & Vocal Calibration System | Najran University Multimodal Capstone Project\n",
    "        </p>\n",
    "    </div>\n",
    '    """)\n\n',
    "    with gr.Row():\n",
    "        with gr.Column(scale=1):\n",
    '            gr.Markdown("### Configuration and Inputs")\n',
    '            consent_checkbox = gr.Checkbox(\n',
    '                label="I agree to let the system process this audio locally for academic evaluation.",\n',
    "                value=False\n",
    "            )\n",
    '            audio_uploader = gr.Audio(\n',
    '                label="Upload Presentation Audio (WAV or MP3)",\n',
    "                type='filepath'\n",
    "            )\n",
    '            submit_button = gr.Button("Analyze Presentation", variant="primary")\n\n',
    "        with gr.Column(scale=2):\n",
    '            gr.Markdown("### System Analytics Output")\n',
    '            timeline_plot = gr.Plot(label="Speech Pace Over Time")\n',
    '            output_report = gr.Markdown("### Waiting for inputs...")\n\n',
    "    submit_button.click(\n",
    "        fn=analyze_presentation,\n",
    "        inputs=[audio_uploader, consent_checkbox],\n",
    "        outputs=[output_report, timeline_plot]\n",
    "    )\n\n",
    'if __name__ == "__main__":\n',
    "    demo.launch()\n"
]

with open("app.py", "w") as app_file:
    app_file.writelines(app_lines)

print("Successfully generated: app.py")
print("\n[Files saved locally! You can now see them in the sidebar file tree]")